# LABORATORIO 09 - Arquitetura RAG Avancada
## HNSW + HyDE (Groq/LLaMA 3.3) + Cross-Encoders

---

**Objetivo:** Construir um pipeline RAG de nivel de producao para busca semantica em manuais medicos tecnicos.

**Problema:** A query *dor de cabeca latejante e luz incomodando* esta longe de *cefaleia pulsatil com fotofobia* no espaco vetorial, mesmo sendo equivalentes.

**Solucao:**
```
Query coloquial -> [HyDE/LLaMA 3.3] -> Doc. Hipotetico -> [HNSW] -> Top-10 -> [Cross-Encoder] -> Top-3
```

| Passo | Tecnica | Funcao |
|:---:|---|---|
| 1 | HNSW (FAISS) | Indexar o corpus como grafo hierarquico |
| 2 | HyDE via Groq/LLaMA 3.3 | Transformar query coloquial em jargao tecnico |
| 3 | Bi-Encoder | Busca rapida - Top-10 candidatos |
| 4 | Cross-Encoder | Re-ranking preciso - Top-3 finais |

---
## Instalacao das Dependencias

> Execute esta celula primeiro e aguarde a instalacao terminar.

In [ ]:
!pip install -q faiss-cpu sentence-transformers groq
print('Dependencias instaladas com sucesso!')

---
## Configuracao da Chave da API (Groq)

A chave e lida com seguranca pelo **Secrets do Colab** - ela nunca aparece no codigo.

**Como configurar:**
1. No menu lateral esquerdo, clique no icone de cadeado **(Secrets)**
2. Clique em **Add new secret**
3. **Name:** `GROQ_API_KEY` - **Value:** sua chave do Groq
4. Ative o toggle **Notebook access** e execute a celula abaixo

> Obtenha sua chave gratuita em https://console.groq.com

In [ ]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
groq_client  = Groq(api_key=GROQ_API_KEY)

print('Chave Groq carregada com sucesso! (via Secrets - nenhuma chave exposta no codigo)')

---
## Imports e Configuracoes Globais

In [ ]:
import numpy as np
from typing import List, Dict
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

EMBEDDING_MODEL     = 'sentence-transformers/all-MiniLM-L6-v2'
CROSS_ENCODER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
GROQ_MODEL          = 'llama-3.3-70b-versatile'

TOP_K_RETRIEVE = 10
TOP_K_RERANK   = 3

print('Imports OK')
print(f'   Bi-Encoder   : {EMBEDDING_MODEL}')
print(f'   Cross-Encoder: {CROSS_ENCODER_MODEL}')
print(f'   LLM (HyDE)   : {GROQ_MODEL} via Groq')
print(f'   Funil        : Top-{TOP_K_RETRIEVE} -> Top-{TOP_K_RERANK}')

---
## PASSO 0 - Corpus Simulado

22 fragmentos de manuais medicos tecnicos cobrindo neurologia, farmacologia, emergencias e protocolos clinicos.

In [ ]:
MEDICAL_CORPUS: List[Dict] = [
    {'id': 0,  'title': 'Manual de Neurologia Clinica - Cap. 3',
     'text': 'Cefaleia pulsatil acompanhada de fotofobia e fonofobia e o quadro classico da enxaqueca (migranea) sem aura. A dor localiza-se unilateralmente na regiao frontotemporal e pode durar entre 4 e 72 horas. O tratamento agudo de primeira linha inclui analgesicos AINEs e triptanos.'},
    {'id': 1,  'title': 'Protocolo de Urgencias Neurologicas',
     'text': 'Cefaleia em trovoada (thunderclap headache) de inicio subito e intensidade maxima em menos de 60 segundos deve ser considerada hemorragia subaracnoidea ate prova em contrario. Solicitar TC de cranio sem contraste imediatamente.'},
    {'id': 2,  'title': 'Manual de Oftalmologia - Sintomas Associados',
     'text': 'Fotofobia, definida como hipersensibilidade dolorosa a luz, pode ser sintoma de meningite, cefaleia em salvas, uveite anterior ou enxaqueca. A avaliacao diferencial inclui inspecao do reflexo pupilar e avaliacao de rigidez de nuca.'},
    {'id': 3,  'title': 'Farmacologia Clinica - Analgesicos',
     'text': 'Os triptanos (sumatriptana, rizatriptana) agem como agonistas seletivos dos receptores 5-HT1B/1D, promovendo vasoconstricao das arterias intracranianas e inibicao da liberacao de neuropeptideos pro-inflamatorios. Contraindicados em doenca cardiovascular isquemica.'},
    {'id': 4,  'title': 'Semiologia Medica - Avaliacao da Dor',
     'text': 'A escala visual analogica (EVA) quantifica a intensidade da dor de 0 a 10. Dores pulsateis ou latejantes correlacionam-se frequentemente com mecanismo vascular, enquanto dores constritivas ou em pressao sugerem cefaleia tensional.'},
    {'id': 5,  'title': 'Manual de Neurologia - Cefaleia Tensional',
     'text': 'A cefaleia tensional episodica caracteriza-se por pressao bilateral, nao pulsatil, de intensidade leve a moderada, sem nauseas e sem agravamento pela atividade fisica rotineira. E o tipo de cefaleia mais prevalente na populacao geral.'},
    {'id': 6,  'title': 'Protocolo Clinico - Meningite Bacteriana',
     'text': 'A triade classica da meningite bacteriana aguda inclui febre alta, rigidez de nuca (meningismo) e alteracao do nivel de consciencia. Cefaleia intensa, fotofobia e vomitos em jato completam o quadro. Puncao lombar e obrigatoria na ausencia de contraindicacoes neurologicas.'},
    {'id': 7,  'title': 'Diretriz de Hipertensao Arterial Sistemica',
     'text': 'A crise hipertensiva (PA > 180/120 mmHg) pode manifestar-se com cefaleia occipital pulsatil, epistaxe, visao turva e dispneia. A emergencia hipertensiva ocorre quando ha lesao de orgao-alvo: encefalopatia, infarto agudo ou disseccao aortica.'},
    {'id': 8,  'title': 'Manual de Otorrinolaringologia - Sinusite',
     'text': 'A sinusite maxilar aguda bacteriana causa dor facial de carater pressivo na regiao malar e frontal, agravada pela posicao ortoestatica. Cefaleia frontal intensa ao inclinar a cabeca para frente e sinal caracteristico de sinusite frontal.'},
    {'id': 9,  'title': 'Neurologia - Cefaleia em Salvas (Cluster Headache)',
     'text': 'A cefaleia em salvas e uma forma de cefaleia trigeminoautonomica caracterizada por dor unilateral periorbital de intensidade excruciante, com duracao de 15 a 180 minutos. Sintomas autonomicos ipsilaterais: lacrimejamento, injecao conjuntival, ptose e rinorreia.'},
    {'id': 10, 'title': 'Manual de Pediatria - Cefaleia na Infancia',
     'text': 'Criancas com cefaleia recorrente devem ser avaliadas para enxaqueca pediatrica, que frequentemente se apresenta de forma bilateral, com episodios mais curtos (1-72 horas). Dor abdominal e cinetose sao comorbidades comuns nesta faixa etaria.'},
    {'id': 11, 'title': 'Tratado de Medicina Interna - Tireoidopatias',
     'text': 'O hipotireoidismo pode causar cefaleia cronica difusa, fadiga intensa, ganho de peso, bradicardia e intolerancia ao frio. O diagnostico baseia-se na dosagem serica de TSH elevado e T4 livre reduzido. Tratamento com levotiroxina sodica.'},
    {'id': 12, 'title': 'Protocolo de Neuroimagem - Indicacoes de TC',
     'text': "Indicacoes absolutas de TC de cranio em cefaleia: inicio subito ('pior dor da vida'), cefaleia progressiva sem melhora, associacao com febre e rigidez de nuca, deficit neurologico focal, papiledema ao fundo de olho ou pos-trauma craniano."},
    {'id': 13, 'title': 'Manual de Anestesiologia - Dor Neuropatica',
     'text': 'A dor neuropatica caracteriza-se por sensacao de queimacao, choque eletrico ou formigamento ao longo de um dermatomero. A neuralgia do trigemeo provoca dores faciais lancinantes unilaterais, geralmente desencadeadas por estimulos tateis leves (allodynia).'},
    {'id': 14, 'title': 'Guia de Psiquiatria Clinica - Transtornos de Ansiedade',
     'text': 'O transtorno de ansiedade generalizada (TAG) frequentemente cursa com cefaleia tensional cronica, insonia, tensao muscular cervical, irritabilidade e dificuldade de concentracao. O tratamento inclui TCC e inibidores seletivos de recaptacao de serotonina (ISRS).'},
    {'id': 15, 'title': 'Semiologia - Exame Neurologico Basico',
     'text': 'O exame neurologico sumario inclui: avaliacao do nivel de consciencia pela Escala de Glasgow, pares cranianos (II a XII), forca motora dos quatro membros, reflexos tendinosos profundos, sensibilidade superficial e profunda e coordenacao cerebelar.'},
    {'id': 16, 'title': 'Manual de Geriatria - Cefaleia no Idoso',
     'text': 'Arterite de celulas gigantes (arterite temporal) deve ser suspeitada em pacientes acima de 50 anos com cefaleia temporal nova, claudicacao de mandibula e VHS elevada. Risco de cegueira por oclusao da arteria oftalmica. Iniciar corticoide empiricamente.'},
    {'id': 17, 'title': 'Farmacologia - Analgesia Escalonada (OMS)',
     'text': 'A escada analgesica da OMS propoe: degrau 1 (dipirona, paracetamol, AINEs para dor leve), degrau 2 (tramadol para dor moderada) e degrau 3 (morfina para dor intensa). Adjuvantes como antidepressivos triciclicos podem ser acrescentados em qualquer degrau.'},
    {'id': 18, 'title': 'Protocolo de Pos-Operatorio Neurocirurgico',
     'text': 'Cefaleia pos-raquianestesia (cefaleia postural) e consequencia do vazamento de LCR pelo orificicio de puncao. Caracteriza-se por dor intensa ao sentar, aliviada em decubito dorsal. Tratamento: repouso, hidratacao, cafeina oral ou blood patch epidural.'},
    {'id': 19, 'title': 'Diretriz de Cefaleia Cronica Diaria',
     'text': 'Cefaleia cronica diaria e definida como dor de cabeca presente em mais de 15 dias por mes durante pelo menos 3 meses. Causas: enxaqueca cronica, cefaleia por uso excessivo de medicamentos (rebote analgesico), cefaleia tensional cronica e hemicrania continua.'},
    {'id': 20, 'title': 'Manual de Emergencias - Hipertensao Intracraniana',
     'text': 'Sindrome de hipertensao intracraniana: cefaleia progressiva matinal, vomitos em jato sem nausea previa, papiledema bilateral e comprometimento progressivo da consciencia. Causas: tumor cerebral, hematoma subdural, abscesso encefalico e hidrocefalia obstrutiva.'},
    {'id': 21, 'title': 'Protocolo de AVC - Janela Terapeutica',
     'text': 'O AVC isquemico agudo pode ser tratado com trombolise intravenosa (alteplase 0,9 mg/kg) dentro de uma janela terapeutica de ate 4,5 horas do inicio dos sintomas. A trombectomia mecanica e indicada para oclusoes de grandes vasos ate 24 horas.'},
]

print(f'Corpus carregado: {len(MEDICAL_CORPUS)} fragmentos de manuais medicos.')

---
## PASSO 1 - Construcao do Indice HNSW com FAISS

### Por que HNSW e nao KNN exato?

O **KNN exato** compara a query contra todos os N vetores com complexidade `O(N x d)`. Para 1 milhao de documentos, isso representa cerca de 500 ms por query.

O **HNSW** constroi um grafo multicamada e percorre do topo ate a base em `O(log N)`, menos de 1 ms por query.

### Hiperparametros e RAM

| Parametro | Papel | Valor |
|---|---|:---:|
| **M** | Conexoes por no. Maior M = mais recall, mais RAM | 32 |
| **ef_construction** | Fila na indexacao. Maior = grafo melhor, build mais lento | 200 |
| **ef_search** | Fila na busca. Ajustavel sem re-indexar | 50 |

**Formula de RAM:**
```
RAM = N x d x 4 bytes (vetores) + N x M x 2 x 8 bytes (ponteiros do grafo)
```
O HNSW usa cerca de 15 a 20% mais RAM que apenas os vetores brutos, mas elimina o custo O(N) de busca.

In [ ]:
print('Carregando modelo de embedding (Bi-Encoder)...')
bi_encoder = SentenceTransformer(EMBEDDING_MODEL)
print(f'Modelo carregado: {EMBEDDING_MODEL}')

print('\nGerando embeddings do corpus...')
texts = [doc['text'] for doc in MEDICAL_CORPUS]
embeddings = bi_encoder.encode(
    texts, show_progress_bar=True, convert_to_numpy=True
).astype('float32')

faiss.normalize_L2(embeddings)

dim             = embeddings.shape[1]
M               = 32
ef_construction = 200
ef_search       = 50

index = faiss.IndexHNSWFlat(dim, M)
index.hnsw.efConstruction = ef_construction
index.hnsw.efSearch       = ef_search
index.add(embeddings)

ram_vetores   = index.ntotal * dim * 4
ram_ponteiros = index.ntotal * M * 2 * 8
ram_total     = ram_vetores + ram_ponteiros

print(f'\n' + '=' * 50)
print(f'  INDICE HNSW CRIADO')
print(f'=' * 50)
print(f'  Vetores indexados : {index.ntotal}')
print(f'  Dimensao          : {dim}')
print(f'  M                 : {M}')
print(f'  ef_construction   : {ef_construction}')
print(f'  ef_search         : {ef_search}')
print(f'  RAM estimada      : {ram_total:,} bytes ({ram_total/1024:.1f} KB)')

---
## PASSO 2 - HyDE: Hypothetical Document Embeddings (Groq / LLaMA 3.3)

### O problema geometrico
```
  Regiao tecnica:                   Regiao coloquial:
  - cefaleia pulsatil                - dor de cabeca latejante
  - fotofobia              <--distancia grande-->
  - migranea ICHD-3
```

### A solucao HyDE
O **LLaMA 3.3** gera uma resposta tecnica hipotetica. Pode ser imprecisa, mas estara no mesmo espaco semantico dos documentos reais, servindo como ancora geometrica para a busca.

> A chave da API e lida dos **Secrets do Colab** - nunca exposta no codigo.

In [ ]:
def generate_hypothetical_document(query: str) -> str:
    """
    Chama o LLaMA 3.3 via Groq API para gerar um documento hipotetico tecnico
    a partir de uma query coloquial do usuario (tecnica HyDE).
    """
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                'role': 'system',
                'content': (
                    'Voce e um medico especialista. Dado um sintoma descrito pelo paciente '
                    'em linguagem coloquial, gere um paragrafo tecnico medico como se fosse '
                    'um trecho de manual clinico, usando jargao medico especializado. '
                    'Responda APENAS com o paragrafo tecnico, sem introducoes ou explicacoes.'
                ),
            },
            {'role': 'user', 'content': query},
        ],
    )
    return response.choices[0].message.content.strip()


USER_QUERY = 'dor de cabeca latejante e luz incomodando'

print('Chamando Groq/LLaMA 3.3 para gerar documento hipotetico...')
hypothetical_doc = generate_hypothetical_document(USER_QUERY)

print('\nHyDE - Transformacao da Query')
print(f'\n  Query coloquial do usuario:')
print(f"  '{USER_QUERY}'")
print(f'\n  Documento hipotetico gerado pelo LLaMA 3.3:')
print(f"  '{hypothetical_doc}'")

---
## PASSO 3 - Busca Rapida via Bi-Encoder + HNSW (Top-10)

O vetor do **documento hipotetico** e usado para buscar os documentos mais proximos no indice HNSW - o funil largo, priorizando alto recall.

In [ ]:
query_vec = bi_encoder.encode(
    [hypothetical_doc], convert_to_numpy=True
).astype('float32')
faiss.normalize_L2(query_vec)

distances, indices_found = index.search(query_vec, TOP_K_RETRIEVE)

candidates = []
for rank, (dist, idx) in enumerate(zip(distances[0], indices_found[0]), 1):
    doc = MEDICAL_CORPUS[idx].copy()
    doc['bi_encoder_score'] = float(dist)
    doc['bi_encoder_rank']  = rank
    candidates.append(doc)

print(f'Top-{TOP_K_RETRIEVE} documentos recuperados pelo Bi-Encoder + HNSW')
print(f'(vetor do documento hipotetico -> indice HNSW)')
print()
print(f"  {'Rank':<5} {'Score':>8}   Titulo")
print('  ' + '-' * 65)
for doc in candidates:
    print(f"  {doc['bi_encoder_rank']:<5} {doc['bi_encoder_score']:>8.4f}   {doc['title']}")

---
## PASSO 4 - Re-ranking com Cross-Encoder (Top-3 Finais)

| | Bi-Encoder | Cross-Encoder |
|---|---|---|
| **Input** | Query e Doc separados | [CLS] Query [SEP] Doc juntos |
| **Atencao cruzada** | Nao tem | Bidirecional completa |
| **Velocidade** | Muito rapida | Lenta (par a par) |
| **Precisao** | ~80-90% recall | ~95-99% precision |
| **Uso** | Funil largo (Top-10) | Funil fino (Top-3) |

In [ ]:
print('Carregando Cross-Encoder...')
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
print(f'Cross-Encoder carregado: {CROSS_ENCODER_MODEL}')

In [ ]:
pairs = [(USER_QUERY, doc['text']) for doc in candidates]

print('Calculando scores do Cross-Encoder...')
ce_scores = cross_encoder.predict(pairs)

for doc, score in zip(candidates, ce_scores):
    doc['cross_encoder_score'] = float(score)

reranked = sorted(candidates, key=lambda d: d['cross_encoder_score'], reverse=True)

print('\nRanking completo apos Cross-Encoder (* = selecionados para o LLM)')
print()
print(f"  {'':2} {'Rank':<5} {'CE Score':>10}  {'BI Score':>10}   Titulo")
print('  ' + '-' * 78)
for rank, doc in enumerate(reranked, 1):
    marker = '*' if rank <= TOP_K_RERANK else ' '
    print(
        f"  {marker} {rank:<4} {doc['cross_encoder_score']:>10.4f}  "
        f"{doc['bi_encoder_score']:>10.4f}   {doc['title']}"
    )

In [ ]:
top_docs = reranked[:TOP_K_RERANK]

print('=' * 65)
print(f'  DOCUMENTOS FINAIS INJETADOS NO CONTEXTO DO LLM (Top-{TOP_K_RERANK})')
print('=' * 65)

for rank, doc in enumerate(top_docs, 1):
    print(f"\n  [{rank}] {doc['title']}")
    print(f"       Cross-Encoder Score : {doc['cross_encoder_score']:.4f}")
    print(f"       Bi-Encoder Score    : {doc['bi_encoder_score']:.4f}")
    print(f"       Texto: {doc['text']}")

---
## BONUS - Testar com Multiplas Queries

Execute o pipeline completo para outras queries coloquiais usando o LLaMA 3.3 real.

In [ ]:
def run_full_pipeline(query):
    print('\n' + '=' * 65)
    print(f"  QUERY: '{query}'")
    print('=' * 65)

    print('  Gerando documento hipotetico via LLaMA 3.3...')
    hyp_doc = generate_hypothetical_document(query)
    print(f"  Doc. hipotetico: '{hyp_doc[:120]}...")

    q_vec = bi_encoder.encode([hyp_doc], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q_vec)
    dists, idxs = index.search(q_vec, TOP_K_RETRIEVE)

    cands = []
    for r, (d, i) in enumerate(zip(dists[0], idxs[0]), 1):
        doc = MEDICAL_CORPUS[i].copy()
        doc['bi_encoder_score'] = float(d)
        cands.append(doc)

    pairs  = [(query, doc['text']) for doc in cands]
    scores = cross_encoder.predict(pairs)
    for doc, s in zip(cands, scores):
        doc['cross_encoder_score'] = float(s)

    reranked = sorted(cands, key=lambda d: d['cross_encoder_score'], reverse=True)

    print(f'\n  Top-{TOP_K_RERANK} documentos selecionados:')
    for rank, doc in enumerate(reranked[:TOP_K_RERANK], 1):
        print(f"  [{rank}] (CE={doc['cross_encoder_score']:.3f}) {doc['title']}")


outras_queries = [
    'pressao alta dor na cabeca',
    'dor de cabeca todo dia ha meses',
    'tontura e vomito com ouvido tampado',
]

for q in outras_queries:
    run_full_pipeline(q)

---
## Resumo do Pipeline

```
Usuario digita query coloquial
         |
         v
+------------------------------------------+
|  PASSO 2 - HyDE (Groq / LLaMA 3.3)      |
|  LLM gera resposta tecnica hipotetica    |
|  -> resolve a lacuna semantica           |
+------------------------------------------+
         |  vetor do documento hipotetico
         v
+------------------------------------------+
|  PASSO 3 - Bi-Encoder + HNSW            |
|  Busca O(log N) no grafo hierarquico     |
|  -> Top-10 candidatos (funil largo)      |
+------------------------------------------+
         |  10 documentos candidatos
         v
+------------------------------------------+
|  PASSO 4 - Cross-Encoder                |
|  Atencao bidirecional (query + doc)      |
|  -> Top-3 documentos finais (funil fino) |
+------------------------------------------+
         |
         v
   Contexto injetado no LLM gerador
   -> Resposta final ao usuario
```

| Etapa | Tecnica | Velocidade | Precisao |
|---|---|:---:|:---:|
| Transformacao | HyDE (LLaMA 3.3 / Groq) | Rapida | Resolve lacuna semantica |
| Recuperacao | Bi-Encoder + HNSW | O(log N) | Alta recall |
| Refinamento | Cross-Encoder | O(K) | Alta precision |

---
> Partes geradas/complementadas com IA, revisadas por Ingrid.